# NeuroLens

**An interactive neuroscience playground built on TRIBE v2**

Explore how the brain responds to video, audio, and text — and discover which AI models think most like humans.

Three modules:
1. **PREDICT** — See predicted brain activation for a stimulus
2. **MATCH** — Find content that triggers specific brain states
3. **EVAL** — Benchmark AI models against biological brain responses

In [ ]:
import sys, os
from pathlib import Path

# Clone repo and set up environment (Colab)
if not os.path.exists('neurolens'):
    !git clone https://github.com/ansumanss/tribev2.git
    os.chdir('tribev2')

sys.path.insert(0, os.getcwd())

# Install dependencies if missing
try:
    import nilearn, plotly, ipywidgets
except ImportError:
    !pip install -q -e '.[plotting]' plotly ipywidgets scipy

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import json as _json
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Generate demo cache if no real cache exists
CACHE_DIR = Path('neurolens_cache')
if not (CACHE_DIR / 'stimuli' / 'metadata.json').exists():
    print('No cache found. Generating demo cache with synthetic data...')
    for d in ['stimuli', 'brain_preds', 'roi_summaries']:
        (CACHE_DIR / d).mkdir(parents=True, exist_ok=True)
    for model in ['clip', 'whisper', 'gpt2']:
        (CACHE_DIR / 'embeddings' / model).mkdir(parents=True, exist_ok=True)

    demo_stimuli = [
        {'id': 'demo_001', 'name': 'Nature scene', 'category': 'Silence + Visuals', 'media_type': 'video', 'duration_sec': 10.0},
        {'id': 'demo_002', 'name': 'Speech excerpt', 'category': 'Speech', 'media_type': 'video', 'duration_sec': 12.0},
        {'id': 'demo_003', 'name': 'Classical music', 'category': 'Music', 'media_type': 'audio', 'duration_sec': 15.0},
        {'id': 'demo_004', 'name': 'Horror clip', 'category': 'Emotional', 'media_type': 'video', 'duration_sec': 8.0},
        {'id': 'demo_005', 'name': 'Movie scene', 'category': 'Multimodal-rich', 'media_type': 'video', 'duration_sec': 10.0},
    ]
    (CACHE_DIR / 'stimuli' / 'metadata.json').write_text(_json.dumps(demo_stimuli, indent=2))

    np.random.seed(42)
    roi_names = ['Visual Cortex', 'Auditory Cortex', 'Language Areas', 'Motor Cortex',
                 'Prefrontal Cortex', 'Temporal Cortex', 'Parietal Cortex',
                 'Somatosensory Cortex', 'Face-Selective Areas']
    for s in demo_stimuli:
        sid = s['id']
        preds = np.random.randn(8, 20484).astype(np.float32)
        np.savez(CACHE_DIR / 'brain_preds' / f'{sid}.npz', preds=preds)
        roi = {name: round(float(np.random.rand()), 3) for name in roi_names}
        (CACHE_DIR / 'roi_summaries' / f'{sid}.json').write_text(_json.dumps(roi))
        for model in ['clip', 'whisper', 'gpt2']:
            torch.save(torch.randn(256), CACHE_DIR / 'embeddings' / model / f'{sid}.pt')
    print(f'Demo cache ready: {len(demo_stimuli)} stimuli, 3 models')
    print('Note: Using synthetic data. Run neurolens_precompute.ipynb for real brain predictions.\n')

from neurolens.cache import CacheManager
from neurolens.stimulus import StimulusLibrary
from neurolens.predict import get_prediction_at_time, get_num_timesteps, get_top_rois
from neurolens.match import find_similar_stimuli, build_target_from_regions, find_contrast_stimuli
from neurolens.eval import compute_all_model_alignments, compute_model_brain_alignment
from neurolens.roi import get_roi_group_names, ROI_GROUPS
from neurolens.viz import plot_brain_surface, make_radar_chart

cache = CacheManager(CACHE_DIR)
library = StimulusLibrary(CACHE_DIR)
print(f'Loaded {len(library)} stimuli, {len(cache.available_models())} models')

---
## Module 1: PREDICT
Select a stimulus and explore how it activates different brain regions.

In [ ]:
# Stimulus picker
stim_dropdown = widgets.Dropdown(
    options=library.dropdown_options(),
    description="Stimulus:",
    style={"description_width": "initial"},
)

# Time slider (updated dynamically)
time_slider = widgets.IntSlider(
    value=0, min=0, max=1, step=1,
    description="Timestep:",
    continuous_update=False,
)

# View selector
view_select = widgets.SelectMultiple(
    options=["left", "right", "medial_left", "medial_right", "dorsal"],
    value=["left", "right"],
    description="Views:",
)

output_predict = widgets.Output()

def update_predict(*args):
    sid = stim_dropdown.value
    n_steps = get_num_timesteps(cache, sid)
    time_slider.max = n_steps - 1

    with output_predict:
        clear_output(wait=True)
        data = get_prediction_at_time(cache, sid, time_slider.value)
        stim = library.get(sid)
        fig = plot_brain_surface(
            data,
            views=list(view_select.value),
            title=f"{stim.name} (t={time_slider.value})",
        )
        plt.show()

        # Top ROIs
        top = get_top_rois(cache, sid, k=5)
        print("\nTop activated regions:")
        for name, val in top:
            bar = "\u2588" * int(abs(val) * 20)
            print(f"  {name:.<30s} {val:+.3f} {bar}")

stim_dropdown.observe(update_predict, names="value")
time_slider.observe(update_predict, names="value")
view_select.observe(update_predict, names="value")

display(widgets.VBox([
    widgets.HBox([stim_dropdown, time_slider]),
    view_select,
    output_predict,
]))
update_predict()

---
## Module 2: MATCH
Find content that activates specific brain regions, or discover neurally similar stimuli.

In [ ]:
# Tab 1: Region picker
match_mode = widgets.ToggleButtons(
    options=["Region Picker", "More Like This", "Contrast"],
    description="Mode:",
)

# Region picker controls
region_dropdowns = {}
for name in get_roi_group_names():
    region_dropdowns[name] = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.1,
        description=name, style={"description_width": "initial"},
        layout=widgets.Layout(width="400px"),
    )
region_box = widgets.VBox(list(region_dropdowns.values()))

# More Like This controls
source_dropdown = widgets.Dropdown(
    options=library.dropdown_options(),
    description="Source:",
    style={"description_width": "initial"},
)

# Contrast controls
max_roi = widgets.Dropdown(options=get_roi_group_names(), description="Maximize:")
min_roi = widgets.Dropdown(options=get_roi_group_names(), description="Minimize:", value=get_roi_group_names()[1])

output_match = widgets.Output()

def run_match(btn=None):
    with output_match:
        clear_output(wait=True)
        ids = library.ids()
        mode = match_mode.value

        if mode == "Region Picker":
            intensities = {name: slider.value for name, slider in region_dropdowns.items()}
            target = build_target_from_regions(intensities)
            results = find_similar_stimuli(cache, target, ids, top_k=5)
        elif mode == "More Like This":
            source_preds = cache.load_brain_preds(source_dropdown.value)
            target = source_preds.mean(axis=0)
            results = find_similar_stimuli(cache, target, ids, top_k=5)
        else:  # Contrast
            results = find_contrast_stimuli(
                cache, ids, max_roi.value, min_roi.value, top_k=5
            )

        print(f"Top matches ({mode}):\n")
        radar_data = {}
        for rank, (sid, score) in enumerate(results, 1):
            stim = library.get(sid)
            print(f"  {rank}. {stim.name} [{stim.category}] \u2014 score: {score:.3f}")
            roi = cache.load_roi_summary(sid)
            if roi and rank <= 3:
                radar_data[stim.name] = roi

        if radar_data:
            fig = make_radar_chart(radar_data, title="ROI Activation Profiles")
            plt.show()

match_btn = widgets.Button(description="Find Matches", button_style="primary")
match_btn.on_click(run_match)

display(widgets.VBox([
    match_mode,
    region_box,
    widgets.HBox([source_dropdown]),
    widgets.HBox([max_roi, min_roi]),
    match_btn,
    output_match,
]))

---
## Module 3: EVAL
Which AI model thinks most like a human brain?
Compare model representations against predicted brain responses.

In [ ]:
output_eval = widgets.Output()

def run_eval(btn=None):
    with output_eval:
        clear_output(wait=True)
        ids = library.ids()
        print("Computing brain alignment scores (RSA)...\n")

        scores = compute_all_model_alignments(cache, ids)

        # Leaderboard
        print("=" * 50)
        print(f"{'Rank':<6}{'Model':<20}{'Brain Alignment':>15}")
        print("=" * 50)
        for rank, (model, score) in enumerate(scores.items(), 1):
            bar = "\u2588" * int(max(0, score) * 30)
            print(f"{rank:<6}{model:<20}{score:>+.4f}  {bar}")
        print("=" * 50)

        if len(scores) >= 2:
            print("\nSelect two models to compare:")

model_a = widgets.Dropdown(
    options=cache.available_models(),
    description="Model A:",
)
model_b = widgets.Dropdown(
    options=cache.available_models(),
    description="Model B:",
    value=cache.available_models()[-1] if len(cache.available_models()) > 1 else cache.available_models()[0],
)

output_compare = widgets.Output()

def compare_models(btn=None):
    with output_compare:
        clear_output(wait=True)
        ids = library.ids()
        print(f"Comparing {model_a.value} vs {model_b.value}...\n")
        score_a = compute_model_brain_alignment(cache, model_a.value, ids)
        score_b = compute_model_brain_alignment(cache, model_b.value, ids)

        print(f"  {model_a.value}: RSA = {score_a:+.4f}")
        print(f"  {model_b.value}: RSA = {score_b:+.4f}")

        for model_name in [model_a.value, model_b.value]:
            print(f"\n--- Brain Report Card: {model_name} ---")
            score = compute_model_brain_alignment(cache, model_name, ids)
            pct = max(0, score) * 100
            print(f"  Overall brain alignment: {pct:.1f}%")

compare_btn = widgets.Button(description="Compare Models", button_style="info")
compare_btn.on_click(compare_models)

eval_btn = widgets.Button(description="Run Leaderboard", button_style="primary")
eval_btn.on_click(run_eval)

display(widgets.VBox([
    eval_btn,
    output_eval,
    widgets.HBox([model_a, model_b, compare_btn]),
    output_compare,
]))

---
## Explore Further

- **TRIBE v2 Paper:** [A Foundation Model of Vision, Audition, and Language for In-Silico Neuroscience](https://ai.meta.com/research/publications/a-foundation-model-of-vision-audition-and-language-for-in-silico-neuroscience/)
- **TRIBE v2 Demo:** [aidemos.atmeta.com/tribev2](https://aidemos.atmeta.com/tribev2/)
- **Add more stimuli:** Edit `neurolens_precompute.ipynb` and re-run
- **Add more models:** Extract embeddings from any HuggingFace model into the cache

Built with NeuroLens by Ansuman SS Bhujabala.